In [ ]:
# ============================================================
# FULL RAG PIPELINE: Load -> Split -> Embed -> Store -> Search
# ============================================================
# EMBEDDING METHOD: SentenceTransformer (FREE, Local)
#
# This cell uses FREE local embeddings (no API key required)
# For OpenAI embeddings, see the next cell instead
#
# Pipeline Steps:
#   0. Setup (Model + Database)
#   1. Load (Read documents from disk)
#   2. Split (Break into chunks)
#   3. Embed (Convert to vectors with SentenceTransformer)
#   4. Store (Save in vector database)
# ============================================================

# ============================================================
# IMPORTS
# ============================================================
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from sentence_transformers import SentenceTransformer


# ============================================================
# STEP 0: SETUP - Initialize Model & Database Connection
# ============================================================
print("=" * 70)
print("STEP 0: SETUP - Initialize Model & Database Connection")
print("=" * 70)

# Load the SentenceTransformer model (FREE, runs locally)
model = SentenceTransformer("all-MiniLM-L6-v2")
EMBEDDING_DIM = 384  # SentenceTransformer output size
print("[SUCCESS] Model loaded: all-MiniLM-L6-v2 (384 dimensions)")


    
# Connect to Qdrant Cloud
QDRANT_URL = ""
QDRANT_API_KEY = ""
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
print("[SUCCESS] Connected to Qdrant Cloud")


# ============================================================
# STEP 1: LOAD - Read Documents from Disk
# ============================================================
print("\n" + "=" * 70)
print("STEP 1: LOAD - Read Documents from Disk")
print("=" * 70)

loader = TextLoader("data/nlp_article.txt", encoding="utf-8")
raw_documents = loader.load()

print(f"[SUCCESS] Loaded: {len(raw_documents)} document(s)")
print(f"Preview: {raw_documents[0].page_content[:100]}...")


# ============================================================
# STEP 2: SPLIT - Break Documents into Chunks
# ============================================================
print("\n" + "=" * 70)
print("STEP 2: SPLIT - Break Documents into Chunks")
print("=" * 70)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)
chunks = splitter.split_documents(raw_documents)

print(f"[SUCCESS] Split into: {len(chunks)} chunks")
print(f"Chunk size: 500 characters")
print(f"Chunk overlap: 50 characters")
print(f"First chunk preview: {chunks[0].page_content[:80]}...")


# ============================================================
# STEP 3: EMBED - Convert Text to Vectors (SentenceTransformer)
# ============================================================
print("\n" + "=" * 70)
print("STEP 3: EMBED - Convert Text to Vectors")
print("=" * 70)

chunk_texts = [chunk.page_content for chunk in chunks]
chunk_embeddings = model.encode(chunk_texts)

print(f"[SUCCESS] Embedded: {chunk_embeddings.shape[0]} chunks")
print(f"Vector dimensions: {chunk_embeddings.shape[1]}")
print(f"Model: all-MiniLM-L6-v2 (SentenceTransformer)")


# ============================================================
# STEP 4: STORE - Save Vectors in Qdrant Cloud
# ============================================================
print("\n" + "=" * 70)
print("STEP 4: STORE - Save Vectors in Qdrant Cloud")
print("=" * 70)

# Delete existing collection if it exists
try:
    client.delete_collection(collection_name="nlp_knowledge")
    print("[INFO] Deleted existing 'nlp_knowledge' collection")
except:
    print("[INFO] No existing collection to delete")

# Create new collection
client.create_collection(
    collection_name="nlp_knowledge",
    vectors_config=VectorParams(
        size=EMBEDDING_DIM,      # 384 dimensions
        distance=Distance.COSINE,
    ),
)
print(f"[SUCCESS] Created new collection: 'nlp_knowledge' ({EMBEDDING_DIM} dimensions)")

# Build and upload points
points = []
for i, (text, embedding) in enumerate(zip(chunk_texts, chunk_embeddings)):
    points.append(
        PointStruct(
            id=i,
            vector=embedding.tolist(),
            payload={
                "text": text,
                "source": "nlp_article.txt",
                "chunk_index": i,
            },
        )
    )

client.upsert(collection_name="nlp_knowledge", points=points)
print(f"[SUCCESS] Upserted {len(points)} points to Qdrant Cloud")

info = client.get_collection("nlp_knowledge")
print(f"[SUCCESS] Collection 'nlp_knowledge' now contains: {info.points_count} points")


# ============================================================
# PIPELINE COMPLETE!
# ============================================================
print("\n" + "=" * 70)
print("PIPELINE COMPLETE! You have built a full RAG knowledge base!")
print("=" * 70)
print("\nSummary of what you built:")
print(f"  [1] Loaded a text document from disk")
print(f"  [2] Split it into {len(chunks)} searchable chunks")
print(f"  [3] Converted each chunk into a {EMBEDDING_DIM}-dimensional vector (FREE)")
print(f"  [4] Stored everything in Qdrant Cloud with metadata")
print(f"  [5] Ready to search by meaning in the next cell")
print("\nThis is EXACTLY how production RAG systems work!")
print("\nNext steps:")
print("  - Visit your Qdrant Cloud dashboard to explore the collection")
print("  - Run the next cell to search by meaning")
print("=" * 70)

c:\Users\Dell\.conda\envs\hts_rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


STEP 0: SETUP - Initialize Model & Database Connection


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6205.75it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[SUCCESS] Model loaded: all-MiniLM-L6-v2 (384 dimensions)
[SUCCESS] Connected to Qdrant Cloud

STEP 1: LOAD - Read Documents from Disk
[SUCCESS] Loaded: 1 document(s)
Preview: Natural Language Processing: A Comprehensive Guide

Chapter 1: What is Natural Language Processing?
...

STEP 2: SPLIT - Break Documents into Chunks
[SUCCESS] Split into: 21 chunks
Chunk size: 500 characters
Chunk overlap: 50 characters
First chunk preview: Natural Language Processing: A Comprehensive Guide

Chapter 1: What is Natural L...

STEP 3: EMBED - Convert Text to Vectors
[SUCCESS] Embedded: 21 chunks
Vector dimensions: 384
Model: all-MiniLM-L6-v2 (SentenceTransformer)

STEP 4: STORE - Save Vectors in Qdrant Cloud
[INFO] Deleted existing 'nlp_knowledge' collection
[SUCCESS] Created new collection: 'nlp_knowledge' (384 dimensions)
[SUCCESS] Upserted 21 points to Qdrant Cloud
[SUCCESS] Collection 'nlp_knowledge' now contains: 21 points

PIPELINE COMPLETE! You have built a full RAG knowledge base!

Summary o

In [ ]:
# Using Opne AI embeddings instead of SentenceTransformer (requires API key) (paid)
# ============================================================
# FULL RAG PIPELINE: Load -> Split -> Embed -> Store -> Search
# ============================================================
# EMBEDDING METHOD: OpenAI Embeddings (PAID, requires API key)
#
# This cell uses OpenAI embeddings (requires API key and costs money)
# For FREE local embeddings, use the previous cell instead
#
# Pipeline Steps:
#   0. Setup (OpenAI Client + Database)
#   1. Load (Read documents from disk)
#   2. Split (Break into chunks)
#   3. Embed (Convert to vectors with OpenAI API)
#   4. Store (Save in vector database)
#
# IMPORTANT: You need to set your OpenAI API key below!
# ============================================================

# ============================================================
# IMPORTS
# ============================================================
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from openai import OpenAI
import numpy as np
import os


# ============================================================
# STEP 0: SETUP - Initialize OpenAI Client & Database Connection
# ============================================================
print("=" * 70)
print("STEP 0: SETUP - Initialize OpenAI Client & Database Connection")
print("=" * 70)

# Set your OpenAI API key
# Get your API key from: https://platform.openai.com/api-keys
os.environ["OPENAI_API_KEY"] = ""  # Replace with your actual key
openai_client = OpenAI()

EMBEDDING_DIM = 1536  # OpenAI text-embedding-3-small output size
print("[SUCCESS] OpenAI client initialized (1536 dimensions)")
print("[INFO] Model: text-embedding-3-small")

# Connect to Qdrant Cloud
QDRANT_URL = ""
QDRANT_API_KEY = ""
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
print("[SUCCESS] Connected to Qdrant Cloud")


# ============================================================
# STEP 1: LOAD - Read Documents from Disk
# ============================================================
print("\n" + "=" * 70)
print("STEP 1: LOAD - Read Documents from Disk")
print("=" * 70)

loader = TextLoader("data/nlp_article.txt", encoding="utf-8")
raw_documents = loader.load()

print(f"[SUCCESS] Loaded: {len(raw_documents)} document(s)")
print(f"Preview: {raw_documents[0].page_content[:100]}...")


# ============================================================
# STEP 2: SPLIT - Break Documents into Chunks
# ============================================================
print("\n" + "=" * 70)
print("STEP 2: SPLIT - Break Documents into Chunks")
print("=" * 70)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)
chunks = splitter.split_documents(raw_documents)

print(f"[SUCCESS] Split into: {len(chunks)} chunks")
print(f"Chunk size: 500 characters")
print(f"Chunk overlap: 50 characters")
print(f"First chunk preview: {chunks[0].page_content[:80]}...")


# ============================================================
# STEP 3: EMBED - Convert Text to Vectors (OpenAI API)
# ============================================================
print("\n" + "=" * 70)
print("STEP 3: EMBED - Convert Text to Vectors")
print("=" * 70)

chunk_texts = [chunk.page_content for chunk in chunks]

# Function to get OpenAI embeddings
def get_openai_embedding(text):
    """Get embedding from OpenAI API."""
    response = openai_client.embeddings.create(
        model="text-embedding-3-small",  # or use "text-embedding-ada-002"
        input=text
    )
    return response.data[0].embedding

# Embed all chunks using OpenAI API
chunk_embeddings = []
print(f"[INFO] Embedding {len(chunk_texts)} chunks using OpenAI API...")

for i, text in enumerate(chunk_texts):
    embedding = get_openai_embedding(text)
    chunk_embeddings.append(embedding)
    
    # Progress update every 5 chunks
    if (i + 1) % 5 == 0 or (i + 1) == len(chunk_texts):
        print(f"[PROGRESS] Embedded {i + 1}/{len(chunk_texts)} chunks")

# Convert to numpy array for consistency
chunk_embeddings = np.array(chunk_embeddings)

print(f"[SUCCESS] Embedded: {chunk_embeddings.shape[0]} chunks")
print(f"Vector dimensions: {chunk_embeddings.shape[1]}")
print(f"Model: OpenAI text-embedding-3-small")


# ============================================================
# STEP 4: STORE - Save Vectors in Qdrant Cloud
# ============================================================
print("\n" + "=" * 70)
print("STEP 4: STORE - Save Vectors in Qdrant Cloud")
print("=" * 70)

# Delete existing collection if it exists
try:
    client.delete_collection(collection_name="nlp_knowledge_openai")
    print("[INFO] Deleted existing 'nlp_knowledge_openai' collection")
except:
    print("[INFO] No existing collection to delete")

# Create new collection with 1536 dimensions (OpenAI size)
client.create_collection(
    collection_name="nlp_knowledge_openai",
    vectors_config=VectorParams(
        size=EMBEDDING_DIM,      # 1536 dimensions for OpenAI
        distance=Distance.COSINE,
    ),
)
print(f"[SUCCESS] Created new collection: 'nlp_knowledge_openai' ({EMBEDDING_DIM} dimensions)")

# Build and upload points
points = []
for i, (text, embedding) in enumerate(zip(chunk_texts, chunk_embeddings)):
    points.append(
        PointStruct(
            id=i,
            vector=embedding.tolist(),
            payload={
                "text": text,
                "source": "nlp_article.txt",
                "chunk_index": i,
            },
        )
    )

client.upsert(collection_name="nlp_knowledge_openai", points=points)
print(f"[SUCCESS] Upserted {len(points)} points to Qdrant Cloud")

info = client.get_collection("nlp_knowledge_openai")
print(f"[SUCCESS] Collection 'nlp_knowledge_openai' now contains: {info.points_count} points")


# ============================================================
# PIPELINE COMPLETE!
# ============================================================
print("\n" + "=" * 70)
print("PIPELINE COMPLETE! You have built a full RAG knowledge base!")
print("=" * 70)
print("\nSummary of what you built:")
print(f"  [1] Loaded a text document from disk")
print(f"  [2] Split it into {len(chunks)} searchable chunks")
print(f"  [3] Converted each chunk into a {EMBEDDING_DIM}-dimensional vector (OpenAI)")
print(f"  [4] Stored everything in Qdrant Cloud with metadata")
print(f"  [5] Ready to search by meaning in the next cell")
print("\nThis is EXACTLY how production RAG systems work!")
print("\nComparison with SentenceTransformer:")
print("  - OpenAI: Higher quality, 1536 dims, requires API key, costs money")
print("  - SentenceTransformer: Free, 384 dims, runs locally, good for learning")
print("\nNext steps:")
print("  - Visit your Qdrant Cloud dashboard to explore the collection")
print("  - Compare search quality between OpenAI and SentenceTransformer embeddings")
print("=" * 70)